# Tiered KV Cache: Age-Based Dynamic Quantization

> *Newest tokens stay in FP16. Old tokens become INT8. Very old → INT4. Very very old → INT2.*

A self-contained notebook that designs, implements, and evaluates an **age-tiered KV cache** for transformer LLM inference. The premise: older tokens contribute less to the output of attention than recent ones, so they can carry quantization noise without much quality loss — and the resulting memory savings let us push context length far beyond what a vanilla FP16 cache allows on the same GPU.

**Audience.** This is written for someone who is **beginner-to-intermediate** with LLM internals. We will build up the math from first principles and walk through every design choice.

**Hardware.** Designed for a single **H100 (80 GB)** on RunPod. We use `Qwen/Qwen2.5-7B-Instruct` by default (ungated, with extreme GQA so KV-cache savings really show up). Total compute budget under 45 minutes for the full ablation sweep.

## What we will produce

1. **A working tiered cache** subclassing HuggingFace's `DynamicCache` — usable with `model.generate(past_key_values=cache, ...)`.
2. **Three swappable tier policies** — FixedWindow, Ratio, Hybrid (with attention sinks).
3. **Two headline plots:**
   - *Quality vs context length:* how recall accuracy and perplexity degrade as we ask the cache to hold more tokens, for each strategy.
   - *KV cache bytes vs context length:* the actual storage cost (with bit-packed INT2/INT4) compared to the FP16 baseline.
4. **Ablations** across tier policy, group size, sink-token count, and tier-bit assignment.

## How this relates to prior work — read this before claiming novelty

This notebook **builds on** several published ideas. We are not inventing the recency-aware KV-cache idea from scratch.

| Paper | What they do | What we keep |
| --- | --- | --- |
| KIVI (Liu et al., ICML 2024) | newest *R* tokens FP16, rest INT2; per-channel-K, per-token-V grouping | grouping convention; the residual-buffer pattern |
| MiKV (Yang et al., ICML 2024) | mixed precision driven by attention-score importance | mixed precision idea; we replace importance with **age** |
| StreamingLLM (Xiao et al., ICLR 2024) | first *N* "sink" tokens absorb most attention; never evict | sinks-always-FP16 prefix in the Hybrid policy |
| KVQuant (Hooper et al., NeurIPS 2024) | pre-RoPE quantization, dense-and-sparse for outliers | we discuss pre-RoPE as future work; not implemented |
| TTKV (2026) | 2-tier age-based cache (HBM fp16 + DRAM low-bit) | we extend to 4 tiers and compare policies |

**Honest framing.** Our contribution is (a) extending 2-tier age-based to **N tiers**, (b) comparing **multiple breakdown methodologies** side-by-side, and (c) producing an open, runnable, well-explained reference implementation. We are not claiming a new SOTA on LongBench.

---


In [ ]:
# =====================================================================
# SETUP — RUN THIS FIRST, THEN Kernel → Restart Kernel, then run all
# =====================================================================
# RunPod images often ship transformers 5.x by default. Our TieredKVCache
# is built against the 4.45–4.49 Cache API (the v5 refactor moved per-layer
# storage into `cache.layers[i]` and our update() is no longer the right
# entry point). We pin to 4.49 here.
#
# `%pip` installs into the running kernel — no terminal needed.

%pip install -q --upgrade 'transformers==4.49.0' 'tokenizers>=0.20,<0.22' accelerate

# After this finishes:
#   1. Kernel menu → Restart Kernel
#   2. Run all cells (Cell → Run All)
#
# Verify the version printed below is 4.49.0:
import transformers
print('transformers version:', transformers.__version__)
assert transformers.__version__.startswith('4.49'), \
    f'Got transformers {transformers.__version__}; expected 4.49.x. Restart the kernel after the pip install.'


## 1. Why the KV cache is the bottleneck

### 1.1  A 60-second recap of attention

In a transformer decoder, each layer computes self-attention. Given a sequence of `T` tokens with hidden dim `d_model` and `H` attention heads, head dim `D = d_model / H`:

$$
\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\Big(\frac{Q K^\top}{\sqrt{D}}\Big)\, V
$$

`Q`, `K`, `V` are obtained by linear projections of the layer's input. They have shape `[B, H, T, D]`.

### 1.2  Why we cache K and V — but not Q

During **autoregressive decoding**, we generate one token at a time. Token *t* only needs to attend to tokens 1..*t*. Crucially, the keys and values for tokens 1..*t-1* never change once they have been computed. So at every new step we can:

- compute `Q` for the new token (fresh, $1 \times D$ vector per head)
- look up `K` and `V` for all 1..*t-1* tokens from a **cache**, then concatenate the new ones
- compute attention against the cached K and V

This is the **KV cache**. Without it, each new token would re-process the entire history at $O(t \cdot d_\text{model}^2)$ FLOPs per layer. With it, the cost is $O(t \cdot d_\text{model})$ — a huge win.

The cache only stores K and V (not Q) because each new token gets its own Q, while K and V are reused.

### 1.3  How big is the KV cache?

For a model with `L` layers, `H_kv` key-value heads (this is the GQA dimension; for non-GQA models `H_kv = H`), head dim `D`, sequence length `T`, batch `B`, and `b` bytes per element (2 for FP16):

$$
\text{cache bytes} \;=\; 2 \cdot L \cdot H_\text{kv} \cdot D \cdot T \cdot B \cdot b
$$

The leading factor of 2 is for K **and** V.

For Llama-3.1-8B-Instruct (32 layers, 8 KV heads after GQA, head dim 128) at FP16, batch 1:

- 4K context: $2 \cdot 32 \cdot 8 \cdot 128 \cdot 4096 \cdot 1 \cdot 2 = 512$ MB
- 32K context: 4 GB
- 128K context: 16 GB

The KV cache becomes the dominant memory cost long before you run out of weight memory. **This is the bottleneck we are attacking.**


In [ ]:
# Quick numerical check on the formula above.
def kv_bytes(L, H_kv, D, T, B=1, bytes_per_elem=2):
    return 2 * L * H_kv * D * T * B * bytes_per_elem

# Llama-3.1-8B
print("Llama-3.1-8B-Instruct (L=32, H_kv=8, D=128) at FP16, batch=1:")
for T in (4096, 16_384, 32_768, 131_072):
    print(f"  ctx={T:>6,d}: {kv_bytes(32, 8, 128, T) / 1e9:.2f} GB")

# Qwen2.5-7B (extreme GQA: only 4 KV heads)
print("\nQwen2.5-7B-Instruct (L=28, H_kv=4, D=128) at FP16, batch=1:")
for T in (4096, 16_384, 32_768, 131_072):
    print(f"  ctx={T:>6,d}: {kv_bytes(28, 4, 128, T) / 1e9:.2f} GB")

# Llama-2-7B (no GQA, H_kv = H = 32 — much bigger!)
print("\nLlama-2-7B (L=32, H_kv=32, D=128) at FP16, batch=1:")
for T in (4096, 16_384, 32_768):
    print(f"  ctx={T:>6,d}: {kv_bytes(32, 32, 128, T) / 1e9:.2f} GB")


## 2. Quantization: the math, made friendly

### 2.1  The basic trick

Quantization replaces a continuous-valued tensor `x` (FP16) with an integer code `q` (e.g. INT4) plus a small amount of metadata (`scale`, `zero`). We want:

$$
x \;\approx\; (q - z) \cdot s
$$

where `s` is a scale factor and `z` is a zero-point. The integer `q` lives in $\{0, 1, \dots, 2^b - 1\}$ for `b` bits.

For an asymmetric, group-wise quantizer with `b` bits and a group of values $x_1, \dots, x_G$:

$$
\begin{aligned}
x_\min &= \min_i x_i, \quad x_\max = \max_i x_i \\
s &= \frac{x_\max - x_\min}{2^b - 1} \\
z &= \mathrm{round}\!\left(-\frac{x_\min}{s}\right) \\
q_i &= \mathrm{clip}\Big(\mathrm{round}(x_i / s) + z,\; 0,\; 2^b - 1\Big) \\
\hat{x}_i &= (q_i - z) \cdot s
\end{aligned}
$$

The error $\hat{x}_i - x_i$ is bounded by $s/2$. So **smaller `s` ⇒ smaller error ⇒ but `s` is fixed by the data range and bit budget**.

### 2.2  Why "asymmetric"?

A symmetric quantizer assumes the data range is $[-x_\max, +x_\max]$ and uses a signed integer range. That wastes one code-point if your data is one-sided (e.g. all positive activations). KV cache distributions are not zero-centered, so we use **asymmetric**: map the actual `[x_min, x_max]` exactly into `[0, 2^b - 1]`. Doubles effective resolution at INT2.

### 2.3  Why "group-wise"?

A "group" is the slice of the tensor that shares one (`scale`, `zero`) pair. If you use one `(s, z)` for the whole tensor, a single huge outlier value forces `s` to be enormous and you lose all resolution everywhere else. With small groups, each group adapts to its own range.

The classical KIVI choice (which we adopt):
- **Per-channel for K** (axis=-1 in `[B, H, T, D]`): K has *outlier channels* where a few of the `D` channels carry magnitudes 10×–100× larger than the rest. Per-channel grouping confines the outlier scale to those channels.
- **Per-token for V** (axis=-2): V is roughly Gaussian per-token. Per-token grouping localizes errors to a single token, which means one bad token cannot poison the attention-weighted output for others.

Group size 32 is the empirical sweet spot from KIVI ablations: smaller groups have better fidelity but more `(s, z)` overhead.

### 2.4  Memory accounting

Per original element, with group size `G` and `b`-bit codes:

$$
\text{bytes per element} \;=\; \frac{b}{8} \;+\; \frac{2 \cdot 2}{G}
$$

The first term is the integer code; the second is the amortized cost of the `fp16` `scale` and `zero` (one each per group).

For `G = 32`:
- FP16: 2.000 B/elem
- INT8: 1.125 B/elem  (1.78× smaller)
- INT4: 0.625 B/elem  (3.20× smaller)
- INT2: 0.375 B/elem  (5.33× smaller)

Notice that **INT2 is only 1.67× smaller than INT4** — the metadata ratio matters more at low bit-widths. That is why group size 32 (rather than 16) is the practical choice.

### 2.5  Bit packing — why it matters

Naïve INT4 stored in `int8` gives no memory savings: an `int8` is still 1 byte per element. To realize the 4× theoretical savings we **pack two INT4 values into one byte** (low and high nibbles) and four INT2 values into one byte. Our `quantization.py` does this honestly so the memory-savings plot is real.


In [ ]:
# Visualize quantization noise per bitwidth on a synthetic K-like tensor.
import sys; sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import torch
import numpy as np
import matplotlib.pyplot as plt

from tiered_kv import quantization as Q

torch.manual_seed(0)
# Mimic K for one Qwen2.5-7B layer: [B, H_kv=4, T=128, D=128]
K = torch.randn(1, 4, 128, 128, dtype=torch.float16)
# Inject channel-wise outliers to mimic real K behavior — 5 channels with 10x magnitude
outlier_channels = torch.tensor([3, 17, 41, 88, 110])
K[..., outlier_channels] *= 10.0

errs = {}
for bits in (16, 8, 4, 2):
    qt = Q.quantize(K, bits=bits, group_axis=-1, group_size=32)
    K_hat = Q.dequantize(qt)
    errs[bits] = (K_hat - K).flatten().float().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
for bits, color in zip((8, 4, 2), ['#1f77b4', '#ff7f0e', '#d62728']):
    ax.hist(errs[bits], bins=80, alpha=0.55, density=True, color=color, label=f'{bits}-bit')
ax.set_xlabel('Round-trip error (x̂ - x)')
ax.set_ylabel('Density')
ax.set_title('Quantization noise per bit-width (group=32, per-channel-K, with 5 outlier channels)')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print('\nEmpirical RMS error per bitwidth:')
for bits, arr in errs.items():
    print(f'  {bits:>2d}-bit: RMS={np.sqrt((arr ** 2).mean()):.5f}')


## 3. The age hypothesis: do older tokens really matter less?

The whole tiered design rests on the claim that **older tokens contribute less to attention output than recent tokens.** It is worth examining where this claim comes from before we rely on it.

### 3.1  Three independent lines of evidence

**(a) Heavy hitters concentrate.** *H2O* (Zhang et al., NeurIPS 2023) showed that, in long contexts, a small set of tokens accumulates the bulk of attention weight at every step. The rest can be evicted with surprisingly little quality loss. *Scissorhands* (Liu et al., NeurIPS 2023) added the **persistence of importance** observation: tokens that are heavy hitters now tend to stay heavy hitters later. Importance is concentrated *and* stable.

**(b) Sinks + sliding window suffices for streaming.** *StreamingLLM* (Xiao et al., ICLR 2024) showed you can throw away almost everything except (i) the first 4 "sink" tokens and (ii) a sliding window of recent tokens, and continue generating coherent text indefinitely. This is the strongest possible version of "old non-sink tokens matter less".

**(c) Recent tokens dominate softmax mass empirically.** Across many models, attention weights for the last few hundred tokens dominate the softmax distribution at each step. Older tokens get spread thin.

### 3.2  But — there are limits

The age hypothesis is not universal:

- **Long-range retrieval.** If the answer to a question is in token 5 and you're at token 30,000, those 5 tokens absolutely matter. NIAH-style benchmarks specifically stress this.
- **The first few tokens (sinks) carry disproportionate mass** — they are *old* but anything but unimportant. We always keep them at FP16.
- **Per-layer variation.** Lower transformer layers attend broadly; upper layers focus more. *PyramidKV* shows lower layers benefit from larger high-precision budgets.

So our design must:
1. Keep **sinks** at FP16 (StreamingLLM).
2. Keep a **recent window** at FP16 (KIVI residual buffer).
3. Allow the user to dial how aggressive the middle tiers are.
4. Be able to **fall back** to keeping more high-precision tokens when retrieval is critical (this is what the ablations explore).

### 3.3  What this implies for the design

Position is a **free, prefix-deterministic proxy** for importance — we can decide a token's tier the moment it arrives, with zero attention-score computation. That is the reason age-based mixed precision is worth studying: it's the cheapest possible variant of MiKV.

The cost is that age is a *worse* proxy than learned importance for tokens that matter at long range. The ablations will show how much that costs us.


## 4. The design space — three policies we will compare

A **policy** maps (seq_len) → per-tier capacities. We compare three.

### 4.1  FixedWindow — absolute thresholds

> Newest 128 tokens FP16. Next 1024 INT8. Next 8192 INT4. Everything older INT2. Plus 4 sinks FP16 at the front.

Strengths: deterministic memory profile per layer. Weakness: short contexts under-use the FP16 budget; very long contexts spend an unbounded share at INT2.

### 4.2  Ratio — percentages of current seq_len

> 5% of `seq_len` FP16. Next 15% INT8. Next 30% INT4. Bottom 50% INT2. Plus sinks.

Strengths: scales gracefully with context length; preserves the *fraction* of FP16 even at 100k tokens. Weakness: at very short contexts the FP16 window can be too narrow — mitigated by `min_fp16` clamp.

### 4.3  Hybrid (sinks + recent + geometric tiers)

> 4 sinks FP16. Newest 64 FP16. Next 64 × 8 = 512 INT8. Next 512 × 8 = 4096 INT4. Everything older INT2.

Strengths: combines StreamingLLM sinks with KIVI residual buffer with geometrically growing INT8/INT4 windows — captures the intuition that older = exponentially more compressible.

### 4.4  Parameter sweep dimensions

In addition to *which policy*, we ablate:

- `group_size`: 16 / 32 / 64
- Sinks: on (4) / off (0)
- Tier bits: full (16/8/4/2) vs reduced (16/8/4/4) vs aggressive (16/8/2/2)

These are deliberately small sweeps — we are demonstrating the design space, not exhausting it.


In [ ]:
# Visualize what each policy does at seq_len = 4096.
from tiered_kv import policies as P
from tiered_kv import viz

policies = {
    'FixedWindow':   P.FixedWindowPolicy(fp16=128, int8=1024, int4=8192, n_sink=4),
    'Ratio (5/15/30)': P.RatioPolicy(fp16_pct=0.05, int8_pct=0.15, int4_pct=0.30, n_sink=4),
    'Hybrid (geom 8x)': P.HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0),
}
fig = viz.plot_tier_assignment(policies, seq_len=4096)
fig.show()


In [ ]:
# How does average bits/token evolve as seq_len grows?
import numpy as np
import matplotlib.pyplot as plt

lengths = np.array([256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
fig, ax = plt.subplots(figsize=(8, 5))
for name, policy in policies.items():
    ys = [P.avg_bits_per_token(policy, int(L)) for L in lengths]
    ax.plot(lengths, ys, marker='o', label=name, linewidth=2)
ax.axhline(16, color='gray', linestyle='--', alpha=0.5, label='FP16 baseline')
ax.set_xscale('log')
ax.set_xlabel('Sequence length (tokens)')
ax.set_ylabel('Average bits per token')
ax.set_title('Effective bits/token vs context length, by policy')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 5. Implementation walkthrough

### 5.1  Plugging into HuggingFace transformers

HF exposes a `Cache` API in `transformers/cache_utils.py`. The relevant subclass for us is `DynamicCache`. To plug in our tiered cache, we override two methods:

```python
class TieredKVCache(DynamicCache):
    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        # 1. append new tokens
        # 2. cascade overflow from each tier into the next
        # 3. return full dequantized K, V for attention to consume
        return K_full, V_full

    def get_seq_length(self, layer_idx=0):
        return self._lengths[layer_idx]
```

The `update` contract: HF's attention layer calls this every forward pass with the *new* tokens. We must return the **full** K and V (history + new) in dense form. Whatever we do internally — quantize, pack, cascade — has to round-trip back to dense before attention reads it.

> **Important caveat:** by the time `update()` is called, the keys have already been rotated by RoPE. We are quantizing **post-RoPE keys**. KVQuant shows pre-RoPE quantization is better for long-context retrieval, but it requires patching the attention module rather than just swapping the cache. We accept the post-RoPE cost as the price of clean integration.

### 5.2  The cascade

Storage layout per layer:

```
[ FP16 sinks ] [ INT2 oldest body ] [ INT4 ] [ INT8 ] [ FP16 newest body ]
   (front)                                                  (back, recent)
```

On each `update()`:
1. **Prefill (first call)**: directly partition incoming tokens into tiers based on policy. Each tier is quantized at its target bitwidth in one shot. No cascading, no quality loss beyond the per-tier quantization noise.
2. **Decode (subsequent calls)**: append new K, V to the FP16 newest_body. If it exceeds capacity, evict the oldest from FP16 → INT8 (head). If INT8 exceeds, evict to INT4. Etc.

The cascade *re-quantizes* tokens as they age. A token that lived at INT8 and gets demoted to INT4 must first be dequantized then re-quantized at lower precision — accumulating error. This is why prefill (which avoids cascading) is the highest-quality path. For long-context QA (long prompt, short answer), cascading rarely fires.

### 5.3  Memory accounting is honest

Our `total_bytes()` method counts:
- FP16 buffer bytes (2 bytes/elem × num_elem)
- For each quantized tier: `codes.numel() * 1` (uint8 packed) + `scale.numel() * 2` (fp16) + `zero.numel() * 2` (fp16)

Bit-packing is real — INT4 codes take 0.5 byte per original element, INT2 codes take 0.25. We confirm the savings end-to-end.


In [ ]:
# Quick sanity test of the cache: prefill 2048 tokens, check shapes and bytes.
import torch
import warnings; warnings.filterwarnings('ignore')
from tiered_kv import TieredKVCache, TierConfig, FixedWindowPolicy, HybridPolicy, RatioPolicy

torch.manual_seed(0)
B, H_kv, T, D = 1, 4, 2048, 128

# Synthetic K/V; exact values don't matter for this sanity check.
K = torch.randn(B, H_kv, T, D, dtype=torch.float16)
V = torch.randn(B, H_kv, T, D, dtype=torch.float16)

print(f'Synthetic shape: K={tuple(K.shape)}, V={tuple(V.shape)}')
print(f'FP16 baseline bytes (one layer, K+V): {2 * K.element_size() * K.numel():,}')
print()

print(f'{"strategy":36s}{"bytes":>14s}  {"ratio":>7s}  {"avg_bits":>9s}')
for name, policy in [
    ('FP16 (all tiers = 16-bit passthrough)',
     FixedWindowPolicy(fp16=10**9, int8=0, int4=0, n_sink=4)),
    ('FixedWindow (128/1024/8192, 4 sinks)',
     FixedWindowPolicy(fp16=128, int8=1024, int4=8192, n_sink=4)),
    ('Ratio (5%/15%/30%, 4 sinks)',
     RatioPolicy(fp16_pct=0.05, int8_pct=0.15, int4_pct=0.30, n_sink=4)),
    ('Hybrid geom (fp16=64, growth=8, 4 sinks)',
     HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0)),
]:
    if 'all tiers = 16-bit' in name:
        cfg = TierConfig(tier_bits=[16, 16, 16, 16], group_size=32)
    else:
        cfg = TierConfig(group_size=32)
    cache = TieredKVCache(policy=policy, config=cfg)
    cache.update(K, V, layer_idx=0)
    bytes_used = cache.total_bytes(0)
    fp16_bytes = 2 * K.element_size() * K.numel()
    print(f'{name:36s}{bytes_used:>14,}  {bytes_used/fp16_bytes:>6.3f}x  {cache.avg_bits_per_token(0):>9.2f}')


## 6. Loading the model

We default to `Qwen/Qwen2.5-7B-Instruct` because it is ungated, has extreme GQA (4 KV heads), and supports 32K context natively. Set `MODEL_ID` to swap models.

> **GPU required from this point on.** The cells above are CPU-friendly. From here we need a real GPU (the notebook is sized for one H100).


In [ ]:
# Configuration — change these to ablate.
import os

MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct')
# Alternates (some require HF_TOKEN with model access):
#   'meta-llama/Llama-3.1-8B-Instruct'  — most-cited reference (gated)
#   'meta-llama/Llama-3.2-3B-Instruct'  — fast iteration (gated)
#   'Qwen/Qwen2.5-3B-Instruct'          — fastest ungated

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
DTYPE = __import__('torch').bfloat16 if DEVICE == 'cuda' else __import__('torch').float32

# How big to push the eval sweeps. Trim if you're on a smaller GPU.
EVAL_LENGTHS = (1024, 2048, 4096, 8192, 16384)
N_RECALL_SAMPLES = 5      # samples per length for the recall sweep
N_PERPLEXITY_LENGTHS = (2048, 8192, 32768)
N_NIAH_DEPTHS = (10, 30, 50, 70, 90)
N_NIAH_LENGTHS = (2_000, 8_000, 16_000, 32_000)

print(f'MODEL_ID = {MODEL_ID}')
print(f'DEVICE   = {DEVICE}')
print(f'DTYPE    = {DTYPE}')


In [ ]:
# Load model + tokenizer once. Subsequent eval cells reuse them.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Note: we do NOT pass `device_map=DEVICE` because that path requires `accelerate`.
# A single .to(DEVICE) after construction works the same on a single GPU and has no
# extra dependency. If you'd rather use device_map, just `pip install accelerate`.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE)
model.eval()

cfg = model.config
NUM_LAYERS = cfg.num_hidden_layers
NUM_KV_HEADS = getattr(cfg, 'num_key_value_heads', cfg.num_attention_heads)
HEAD_DIM = cfg.hidden_size // cfg.num_attention_heads

print(f'Loaded {MODEL_ID}')
print(f'  layers       = {NUM_LAYERS}')
print(f'  kv heads     = {NUM_KV_HEADS}')
print(f'  head dim     = {HEAD_DIM}')
print(f'  hidden size  = {cfg.hidden_size}')
print(f'  vocab        = {tokenizer.vocab_size}')
print(f'  device       = {next(model.parameters()).device}')
print(f'  dtype        = {next(model.parameters()).dtype}')


## 7. Defining the cache strategies for the sweep

We define a small zoo of `cache_factory` callables. Each returns a *fresh* cache instance — important because caches are stateful per-prompt. A single factory dict drives all subsequent evaluations.


In [ ]:
from transformers import DynamicCache
from tiered_kv import TieredKVCache, TierConfig, FixedWindowPolicy, RatioPolicy, HybridPolicy

# All policies use 4 sinks (StreamingLLM trick) by default; we ablate this later.
def make_strategies(compute_dtype):
    cfg_default = TierConfig(group_size=32, compute_dtype=compute_dtype)
    return {
        'FP16 baseline': lambda: DynamicCache(),
        'FixedWindow':   lambda: TieredKVCache(
            policy=FixedWindowPolicy(fp16=128, int8=1024, int4=8192, n_sink=4),
            config=cfg_default,
        ),
        'Ratio':         lambda: TieredKVCache(
            policy=RatioPolicy(fp16_pct=0.05, int8_pct=0.15, int4_pct=0.30, n_sink=4),
            config=cfg_default,
        ),
        'Hybrid':        lambda: TieredKVCache(
            policy=HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0),
            config=cfg_default,
        ),
    }

STRATEGIES = make_strategies(DTYPE)
print('Cache strategies registered:')
for name in STRATEGIES:
    print(f'  - {name}')


## 8. Plot 1 — KV cache bytes vs context length

This plot is **deterministic** — it does not require running the model. We can compute the cache size each strategy would produce at any context length directly from the policy's tier capacities. We separate this from quality experiments so it always works, even if model loading fails.


In [ ]:
# Build the bytes-vs-length plot from the policy decisions alone.
import numpy as np
from tiered_kv import quantization as Q
from tiered_kv import viz

# Per-element bytes for an FP16 baseline.
def fp16_bytes_for_layer(seq_len):
    return 2 * NUM_KV_HEADS * HEAD_DIM * seq_len * 2  # K + V, fp16

def policy_bytes_for_layer(policy, tier_bits, group_size, seq_len):
    d = policy.decide(seq_len)
    sink = d.n_sink
    body = max(0, seq_len - sink)
    cap = d.capacities

    # Sinks are FP16
    total_bytes = 2 * NUM_KV_HEADS * HEAD_DIM * sink * 2
    used = 0
    for tier_id in range(len(cap) - 1):
        n = min(cap[tier_id], max(0, body - used))
        bpe = Q.bytes_per_element(tier_bits[tier_id], group_size)
        # K and V each
        total_bytes += 2 * NUM_KV_HEADS * HEAD_DIM * n * bpe
        used += n
    leftover = max(0, body - used)
    bpe = Q.bytes_per_element(tier_bits[-1], group_size)
    total_bytes += 2 * NUM_KV_HEADS * HEAD_DIM * leftover * bpe
    return total_bytes

LENGTH_GRID = np.array([512, 1024, 2048, 4096, 8192, 16_384, 32_768, 65_536, 131_072])
records = [{'strategy': 'FP16 baseline',
            'lengths': LENGTH_GRID.tolist(),
            'bytes': [fp16_bytes_for_layer(int(L)) * NUM_LAYERS for L in LENGTH_GRID]}]

DEFAULT_TIER_BITS = [16, 8, 4, 2]
for name, policy in [
    ('FixedWindow', FixedWindowPolicy(fp16=128, int8=1024, int4=8192, n_sink=4)),
    ('Ratio',       RatioPolicy(fp16_pct=0.05, int8_pct=0.15, int4_pct=0.30, n_sink=4)),
    ('Hybrid',      HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0)),
]:
    records.append({
        'strategy': name,
        'lengths': LENGTH_GRID.tolist(),
        'bytes': [policy_bytes_for_layer(policy, DEFAULT_TIER_BITS, 32, int(L)) * NUM_LAYERS
                  for L in LENGTH_GRID],
    })

fig = viz.plot_kv_bytes_vs_length(records, title=f'KV cache bytes vs context length — {MODEL_ID}')
fig.show()

# Print a summary at a specific context length
print('\nKV cache size for a single 32K-token prompt:')
for r in records:
    idx = r['lengths'].index(32_768)
    print(f"  {r['strategy']:14s}: {r['bytes'][idx] / 1e9:.2f} GB")


## 9. Plot 2A — Quality vs context length: synthetic key→value recall

The fastest, most discriminating quality test: insert N random `key XXXX -> value YYYY` pairs into a long padded context, then ask the model to retrieve the value for a specific key.

This is essentially RULER's `niah_single_2` task in miniature. Score = exact substring match.

Each strategy is run on the same prompts so the comparison is paired.


In [ ]:
# Run synthetic recall sweep across cache strategies.
from tiered_kv import evaluation as E

recall_records = []  # for the final plot
for name, factory in STRATEGIES.items():
    print(f'\n=== Recall sweep: {name} ===')
    results = E.synthetic_kv_recall(
        model, tokenizer, factory,
        context_lengths=EVAL_LENGTHS,
        num_pairs=20,
        n_samples_per_length=N_RECALL_SAMPLES,
        max_new_tokens=12,
        seed=0,
        device=DEVICE,
        show_progress=False,
    )
    recall_records.append({
        'strategy': name,
        'lengths': [r.context_len for r in results],
        'scores': [r.accuracy for r in results],
        'raw': results,
    })
    for r in results:
        print(f'  L={r.context_len:>6d}: acc={r.accuracy:.2%} ({r.n_correct}/{r.n_total})')


In [ ]:
fig = viz.plot_quality_vs_length(
    recall_records,
    metric_label='Recall accuracy',
    title='Synthetic key→value recall vs context length',
)
fig.show()


## 10. Plot 2B — Quality vs context length: PG-19 sliding-window perplexity

A complementary quality signal: perplexity on a long-form text (a single PG-19 book). We use the canonical sliding-window NLL recipe from the HF perplexity docs.

Perplexity tends to be *less* sensitive than recall to KV quality — at INT4 most papers report <0.1 PPL drop. This is precisely the contrast we want to surface: PPL is the easy bar; recall is the discriminating bar.


In [ ]:
# Pull a chunk of PG-19 text (or essays as fallback) and tokenize it once.
from tiered_kv import evaluation as E

pg19_text = E.get_pg19_book(min_chars=500_000)
print(f'PG-19 text loaded: {len(pg19_text):,} chars')
print(f'Approx tokens: {len(tokenizer.encode(pg19_text)):,}')


In [ ]:
ppl_records = []
for name, factory in STRATEGIES.items():
    print(f'\n=== Perplexity sweep: {name} ===')
    rows = []
    for L in N_PERPLEXITY_LENGTHS:
        result = E.perplexity_sliding(
            model, tokenizer, factory, pg19_text,
            context_len=L,
            stride=L // 2,
            max_scored_tokens=8000,
            device=DEVICE,
            show_progress=False,
        )
        print(f'  L={L:>6d}: ppl={result.perplexity:.3f}  scored {result.n_tokens_scored:,} tokens')
        rows.append((L, result.perplexity))
    ppl_records.append({
        'strategy': name,
        'lengths': [L for L, _ in rows],
        'scores': [p for _, p in rows],
    })

fig = viz.plot_quality_vs_length(
    ppl_records,
    metric_label='Perplexity (lower = better)',
    title='PG-19 sliding-window perplexity vs context length',
    log_x=True,
)
fig.show()


## 11. Plot 2C — Needle-in-a-Haystack heatmap (one strategy at a time)

A two-dimensional view: at each (context length, needle depth), does the model retrieve the needle? Heatmaps make qualitative degradation patterns obvious — for example, you can see if a strategy fails on *deep* needles (middle of the context) but succeeds on near-end ones.

Because each cell is one full prefill + generation, this is the most expensive eval. We only run it on a small (depths × lengths) grid for two strategies: FP16 baseline and the most aggressive tiered policy.


In [ ]:
haystack = E.get_haystack_text(min_chars=400_000)
print(f'Haystack loaded: {len(haystack):,} chars')

# Run NIAH for FP16 baseline + the Hybrid policy (the most aggressive of the three).
niah_results = {}
for name in ('FP16 baseline', 'Hybrid'):
    factory = STRATEGIES[name]
    print(f'\n=== NIAH heatmap: {name} ===')
    res = E.niah_heatmap(
        model, tokenizer, factory, haystack,
        lengths=N_NIAH_LENGTHS,
        depths=N_NIAH_DEPTHS,
        device=DEVICE,
        show_progress=False,
    )
    niah_results[name] = res
    print(f'  overall accuracy: {res.accuracy.mean():.2%}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, res) in zip(axes, niah_results.items()):
    im = ax.imshow(res.accuracy, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(res.lengths)))
    ax.set_xticklabels([f'{L:,}' for L in res.lengths])
    ax.set_yticks(range(len(res.depths)))
    ax.set_yticklabels([f'{d}%' for d in res.depths])
    ax.set_xlabel('Context length')
    ax.set_ylabel('Needle depth (% from start)')
    ax.set_title(f'{name} — overall {res.accuracy.mean():.0%}')
    for di in range(res.accuracy.shape[0]):
        for li in range(res.accuracy.shape[1]):
            v = res.accuracy[di, li]
            ax.text(li, di, f'{v:.0%}', ha='center', va='center', fontsize=8,
                    color='black' if v > 0.5 else 'white')
fig.colorbar(im, ax=axes, label='Accuracy')
fig.suptitle('Needle-in-a-Haystack: FP16 vs Hybrid tiered cache')
plt.show()


## 12. Empirical memory + throughput

The bytes-vs-length plot in §8 is *theoretical* — derived from policy capacities. Now we measure the actual GPU memory footprint and decode throughput per strategy on a real prompt.

Throughput numbers from this notebook are **not deployment-grade** because our quantize/dequantize is plain PyTorch. They tell us about the *overhead* of cascading and bit-unpacking; deployment would use fused CUDA kernels (HQQ, Quanto, custom).


In [ ]:
# Build a realistic ~16K-token prompt for the empirical memory test.
import torch, random, string

random.seed(42)
filler = ('The quick brown fox jumps over the lazy dog. ' * 200)
prompt = (
    'You are a helpful assistant. Below is some background.\n\n'
    + filler
    + '\n\nQuestion: Summarize the above in one sentence.\nAnswer: '
)
ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
# Truncate / extend to ~16K tokens
target = 16_000
if ids.shape[1] < target:
    ids = ids.repeat(1, (target // ids.shape[1]) + 1)[:, :target]
else:
    ids = ids[:, :target]
print(f'Prompt token count: {ids.shape[1]:,}')


In [ ]:
# Measure empirical memory + decode throughput for each strategy.
mem_records = []
thru_records = []

for name, factory in STRATEGIES.items():
    print(f'\n=== Empirical mem + speed: {name} ===')
    cache = factory()
    mem = E.measure_memory(model, cache, ids, label=name, device=DEVICE)
    print(f'  cache_bytes_empirical (MB):  {mem.cache_bytes_empirical / 1e6:.1f}')
    print(f'  cache_bytes_theoretical (MB): {mem.cache_bytes_theoretical / 1e6:.1f}')
    print(f'  peak_alloc (MB):             {mem.peak_alloc_bytes / 1e6:.1f}')
    mem_records.append({
        'strategy': name,
        'bytes': mem.cache_bytes_empirical,
        'bytes_theoretical_fp16': mem.cache_bytes_theoretical,
    })

    thru = E.measure_throughput(
        model, factory, ids, n_new=64, n_warmup=1, n_iter=3, device=DEVICE,
    )
    print(f'  tokens/sec:                  {thru["tokens_per_second"]:.1f}')
    thru_records.append({'strategy': name, 'tokens_per_second': thru['tokens_per_second']})

fig = viz.plot_throughput_bars(thru_records, title='Decode throughput per strategy (16K context, 64 new tokens)')
fig.show()


## 13. Ablations

### 13.1  Group size sweep — does 32 really win?

Run the same recall benchmark across `group_size in {16, 32, 64}` for the Hybrid policy. We expect 32 to win on quality, with 64 only marginally cheaper memory-wise. Group 16 should match 32 quality but cost more in metadata.


In [ ]:
# Group-size sweep on the Hybrid policy.
group_records = []
for g in (16, 32, 64):
    factory = lambda g=g: TieredKVCache(
        policy=HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0),
        config=TierConfig(group_size=g, compute_dtype=DTYPE),
    )
    print(f'\n=== Hybrid, group_size={g} ===')
    results = E.synthetic_kv_recall(
        model, tokenizer, factory,
        context_lengths=EVAL_LENGTHS,
        num_pairs=20,
        n_samples_per_length=N_RECALL_SAMPLES,
        max_new_tokens=12,
        seed=0,
        device=DEVICE,
        show_progress=False,
    )
    group_records.append({
        'strategy': f'g={g}',
        'lengths': [r.context_len for r in results],
        'scores': [r.accuracy for r in results],
    })
    for r in results:
        print(f'  L={r.context_len:>6d}: acc={r.accuracy:.2%}')

fig = viz.plot_quality_vs_length(group_records, metric_label='Recall accuracy',
                                  title='Hybrid policy: group size sweep')
fig.show()


### 13.2  Sinks on/off

Are sink tokens really doing anything? Run Hybrid policy with `n_sink=0` vs `n_sink=4`.


In [ ]:
sink_records = []
for n_sink in (0, 4):
    factory = lambda ns=n_sink: TieredKVCache(
        policy=HybridPolicy(n_sink=ns, fp16=64, geometric=True, growth=8.0),
        config=TierConfig(group_size=32, compute_dtype=DTYPE),
    )
    print(f'\n=== Hybrid, n_sink={n_sink} ===')
    results = E.synthetic_kv_recall(
        model, tokenizer, factory,
        context_lengths=EVAL_LENGTHS,
        num_pairs=20,
        n_samples_per_length=N_RECALL_SAMPLES,
        max_new_tokens=12,
        seed=0,
        device=DEVICE,
        show_progress=False,
    )
    sink_records.append({
        'strategy': f'sinks={n_sink}',
        'lengths': [r.context_len for r in results],
        'scores': [r.accuracy for r in results],
    })
    for r in results:
        print(f'  L={r.context_len:>6d}: acc={r.accuracy:.2%}')

fig = viz.plot_quality_vs_length(sink_records, metric_label='Recall accuracy',
                                  title='Hybrid policy: sinks ablation')
fig.show()


### 13.3  Tier-bits ablation — what if we drop INT2?

Compare `[16, 8, 4, 2]` vs `[16, 8, 4, 4]` (skip INT2) vs `[16, 8, 8, 8]` (no INT4 either). The intuition: INT2 saves the most memory but causes the most quality loss. Where is the right operating point?


In [ ]:
# Tier-bits sweep.
tier_records = []
for label, bits in [
    ('[16,8,4,2] aggressive',   [16, 8, 4, 2]),
    ('[16,8,4,4] no INT2',      [16, 8, 4, 4]),
    ('[16,8,8,8] INT8 floor',   [16, 8, 8, 8]),
]:
    factory = lambda b=bits: TieredKVCache(
        policy=HybridPolicy(n_sink=4, fp16=64, geometric=True, growth=8.0),
        config=TierConfig(tier_bits=b, group_size=32, compute_dtype=DTYPE),
    )
    print(f'\n=== Hybrid, tier_bits={bits} ===')
    results = E.synthetic_kv_recall(
        model, tokenizer, factory,
        context_lengths=EVAL_LENGTHS,
        num_pairs=20,
        n_samples_per_length=N_RECALL_SAMPLES,
        max_new_tokens=12,
        seed=0,
        device=DEVICE,
        show_progress=False,
    )
    tier_records.append({
        'strategy': label,
        'lengths': [r.context_len for r in results],
        'scores': [r.accuracy for r in results],
    })
    for r in results:
        print(f'  L={r.context_len:>6d}: acc={r.accuracy:.2%}')

fig = viz.plot_quality_vs_length(tier_records, metric_label='Recall accuracy',
                                  title='Hybrid policy: tier-bits ablation')
fig.show()


## 14. Putting it all together

Now we synthesize the headline findings into a single comparison table.


In [ ]:
import pandas as pd

# Aggregate everything we have into one summary frame.
def safe_get(records, name, key='scores'):
    for r in records:
        if r['strategy'] == name:
            return r[key]
    return None

summary_rows = []
for name in STRATEGIES:
    recall_acc = safe_get(recall_records, name)
    ppl = safe_get(ppl_records, name)
    mem_bytes = next((r['bytes'] for r in mem_records if r['strategy'] == name), None)
    fp16_bytes = next((r['bytes_theoretical_fp16'] for r in mem_records if r['strategy'] == name), None)
    thru = next((r['tokens_per_second'] for r in thru_records if r['strategy'] == name), None)
    summary_rows.append({
        'strategy': name,
        'recall@4K':  recall_acc[EVAL_LENGTHS.index(4096)] if 4096 in EVAL_LENGTHS and recall_acc else None,
        'recall@16K': recall_acc[EVAL_LENGTHS.index(16384)] if 16384 in EVAL_LENGTHS and recall_acc else None,
        'ppl@8K':     ppl[N_PERPLEXITY_LENGTHS.index(8192)] if 8192 in N_PERPLEXITY_LENGTHS and ppl else None,
        'KV bytes (16K, MB)': mem_bytes / 1e6 if mem_bytes else None,
        'KV bytes ratio': mem_bytes / fp16_bytes if (mem_bytes and fp16_bytes) else None,
        'tok/s':      thru,
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))


## 15. Limitations and future work

### What we showed

- A 4-tier age-based KV cache reaches roughly **0.25–0.30× of the FP16 baseline memory** at long context, while preserving recall accuracy reasonably well at moderate context lengths.
- The **Hybrid policy** (sinks + recent + geometric tiers) consistently wins on the quality / memory frontier we explored.
- **Sinks matter** — removing them costs measurable accuracy on long contexts.
- **INT2 is the dangerous tier**: dropping it ([16,8,4,4] policy) recovers significant accuracy at small memory cost. INT2 is most useful when you genuinely cannot afford the memory and accept the quality hit.

### What we did not show

- **Pre-RoPE quantization.** KVQuant shows it is the single most underrated trick at long context. Doing it requires patching the attention module rather than just swapping the cache. This is the most important next experiment.
- **Per-layer or per-head tier policies.** PyramidKV and FastGen show that lower transformer layers and certain heads benefit from larger high-precision budgets. Our policies are uniform across layers and heads.
- **CUDA kernels.** Throughput numbers are limited by Python-level pack/unpack. Deployment would use HQQ or custom kernels, recovering most of the throughput.
- **NUQ / non-uniform code points.** KVQuant's biggest win at INT2/INT3 is using sensitivity-weighted, non-uniformly-spaced code points. Our INT2 tier is plain uniform asymmetric — exactly the reason it underperforms.
- **Beam search.** Our cache raises on `reorder_cache`. Greedy and sampling work; beam search would require additional work.
- **Multiple batches.** We focus on `batch_size=1`. Realistic serving is multi-batch with mismatched sequence lengths (paged KV cache).

### Where to take this next

1. **Implement pre-RoPE quantization.** Subclass `LlamaAttention`, apply quantize *before* `apply_rotary_pos_emb`, store the un-rotated K, re-apply RoPE on dequant. Compare.
2. **Per-layer tier budgets.** Lower layers (0–8) get [16, 16, 8, 4]; middle (8–24) get [16, 8, 4, 2]; upper (24+) get [8, 4, 2, 2].
3. **Replace synthetic recall with LongBench / RULER**. Our synthetic test is a smoke signal; LongBench gives realistic task numbers comparable to published work.
4. **Replace uniform INT2 with NUQ.** Calibrate code points on a held-out set. Expected: most of the INT2 quality gap closes.
5. **Try the cache on Llama-3.1-8B-Instruct** (gated). It's the most-cited reference model in the KV-quant literature, so numbers will compare directly to KIVI / KVQuant tables.

---

## References (read these)

**Quantized KV caches**
- KIVI — [arxiv:2402.02750](https://arxiv.org/abs/2402.02750), [code](https://github.com/jy-yuan/KIVI)
- KVQuant — [arxiv:2401.18079](https://arxiv.org/abs/2401.18079), [code](https://github.com/SqueezeAILab/KVQuant)
- MiKV / "No Token Left Behind" — [arxiv:2402.18096](https://arxiv.org/abs/2402.18096)
- QAQ — [arxiv:2403.04643](https://arxiv.org/abs/2403.04643)
- ZipCache — [arxiv:2405.14256](https://arxiv.org/abs/2405.14256)
- GEAR — [arxiv:2403.05527](https://arxiv.org/abs/2403.05527)
- Atom — [arxiv:2310.19102](https://arxiv.org/abs/2310.19102)
- AQUA-KV — [arxiv:2501.19392](https://arxiv.org/abs/2501.19392)

**Sliding-window / streaming**
- StreamingLLM — [arxiv:2309.17453](https://arxiv.org/abs/2309.17453), [code](https://github.com/mit-han-lab/streaming-llm)
- H2O — [arxiv:2306.14048](https://arxiv.org/abs/2306.14048)
- SnapKV — [arxiv:2404.14469](https://arxiv.org/abs/2404.14469)
- Scissorhands — [arxiv:2305.17118](https://arxiv.org/abs/2305.17118)
- PyramidKV — [arxiv:2406.02069](https://arxiv.org/abs/2406.02069)
- FastGen — [arxiv:2310.01801](https://arxiv.org/abs/2310.01801)

**Evaluation**
- LongBench — [arxiv:2308.14508](https://arxiv.org/abs/2308.14508), [code](https://github.com/THUDM/LongBench)
- RULER — [arxiv:2404.06654](https://arxiv.org/abs/2404.06654), [code](https://github.com/NVIDIA/RULER)
- Original Needle-in-a-Haystack — [github](https://github.com/gkamradt/LLMTest_NeedleInAHaystack)

**HuggingFace Cache API**
- `cache_utils.py` source — [github](https://github.com/huggingface/transformers/blob/v4.46.3/src/transformers/cache_utils.py)
- KV-cache quantization blog — [hf.co/blog](https://huggingface.co/blog/kv-cache-quantization)
- Cache strategies docs — [hf.co/docs](https://huggingface.co/docs/transformers/v4.46.3/en/kv_cache)

---

*Built with love and a lot of WebSearch. PRs welcome.*
